In [84]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# Set up paths
DATA_PROCESSED_PATH = "../data/processed/"
DATA_FEATURES_PATH = "../data/processed/features/"

## 1. Load Processed Data

In [85]:

train_data = pd.read_parquet(f"{DATA_PROCESSED_PATH}train_data_processed.parquet")
test_data = pd.read_parquet(f"{DATA_PROCESSED_PATH}test_data_processed.parquet")

print(f"Train shape: {train_data.shape}")
print(f"Test shape: {test_data.shape}")

# Display key statistics from data prep
print("\nKey Insights from Data Preparation:")
print(f"- Fraud rate: {train_data['isFraud'].mean()*100:.2f}%")
print(f"- Identity coverage: {train_data['id_01'].notna().sum()/len(train_data)*100:.2f}%")
print(f"- Columns with >80% missing: {(train_data.isnull().mean() > 0.8).sum()}")
print(f"- Out of all the features, how many values are missing on average for fraud rows vs. non-fraud rows?")
print(f"- Fraud transactions have {abs(train_data[train_data['isFraud']==1].isnull().mean().mean() - train_data[train_data['isFraud']==0].isnull().mean().mean())*100:.1f}% fewer missing values")

Train shape: (590540, 434)
Test shape: (506691, 433)

Key Insights from Data Preparation:
- Fraud rate: 3.50%
- Identity coverage: 24.42%
- Columns with >80% missing: 74
- Out of all the features, how many values are missing on average for fraud rows vs. non-fraud rows?
- Fraud transactions have 8.1% fewer missing values


In [86]:
print(f"The Shape of the train_data: {train_data.shape}")
print(f"The Shape of the test_data: {test_data.shape}")

The Shape of the train_data: (590540, 434)
The Shape of the test_data: (506691, 433)


In [87]:
# calculates the average missing data rate for not a fraud transaction
train_data[train_data['isFraud']==0].isnull().mean().mean()*100

45.357799296798696

In [88]:
# calculates the average missing data rate for fraud transaction
train_data[train_data['isFraud']==1].isnull().mean().mean()*100

37.25756160246358

## 2. Strategic Feature Selection


In [93]:
# Add only existing columns 
def add_if_exists(feature_list, df):
    return [f for f in feature_list if f in df.columns]

# core_features (as per the EDA)
core_features = [
    'TransactionID', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'isFraud'
]

card_features = [f'card{i}' for i in range(1, 7)]                      # card1–card6
count_features = [f'C{i}' for i in range(1, 15)]                       # C1–C14
time_delta_keep = ['D1', 'D2', 'D3', 'D4', 'D8', 'D10', 'D15']
match_keep = [f'M{i}' for i in range(1, 7)]                            # M1–M6
address_features = ['addr1', 'addr2']
email_features = ['P_emaildomain', 'R_emaildomain']

identity_numeric_keep = [f'id_{i:02d}' for i in range(1, 12)]         # id_01–id_11
identity_categorical_keep = [
    'id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38'
]

device_features = ['DeviceType', 'DeviceInfo']


# Combine all groups
all_feature_groups = (
    core_features +
    card_features +
    count_features +
    time_delta_keep +
    match_keep +
    address_features +
    email_features +
    identity_numeric_keep +
    identity_categorical_keep +
    device_features
)

# --- Filter only features that actually exist ---
features_to_keep = set(add_if_exists(all_feature_groups, train_data))

print(f"Selected {len(features_to_keep)} features to keep.")


Selected 63 features to keep.


### Drop all the low-signal or unimportant features

In [94]:
# Get V-features separately (select top ones)
v_features = [col for col in train_data.columns if col.startswith('V')]

# Remove features not in our keep list (except V-features)
all_features = set(train_data.columns)
features_to_drop = all_features - features_to_keep - set(v_features)

# Remove target from drop list
if 'isFraud' in features_to_drop:
    features_to_drop.remove('isFraud')

print(f"Dropping {len(features_to_drop)} low-signal features")

Dropping 32 low-signal features


In [95]:
# 1. Select features that exist in BOTH train and test
features_common = [f for f in features_to_keep if f in train_data.columns and f in test_data.columns]

# 2. Handle V-features safely
v_features_train = [col for col in train_data.columns if col.startswith('V')]
v_features_test  = [col for col in test_data.columns if col.startswith('V')]

# V-features present in both datasets
v_features_common = list(set(v_features_train) & set(v_features_test))

print(f"Common selected features: {len(features_common)}")
print(f"Common V-features: {len(v_features_common)}")

# 3. Build final feature list (excluding isFraud for test)
selected_features = features_common + v_features_common

# 4. Apply selection
train_data_selected = train_data[selected_features + ['isFraud']].copy()
test_data_selected  = test_data[selected_features].copy()

print("\nAfter safe selection:")
print(f"Train: {train_data_selected.shape}")
print(f"Test : {test_data_selected.shape}")


Common selected features: 43
Common V-features: 339

After safe selection:
Train: (590540, 383)
Test : (506691, 382)


## 3. Select Top V-Features Based on Correlation


In [96]:

def select_top_v_features(train_df, top_n=150, missing_threshold=0.9):
    v_cols = [col for col in train_df.columns if col.startswith('V')]
    
    if not v_cols or 'isFraud' not in train_df.columns:
        return v_cols
    
    # Calculate correlation with target
    v_corr = train_df[v_cols + ['isFraud']].corr()['isFraud'].drop('isFraud', errors='ignore').abs()
    
    # Filter out features with too many missing values
    missing_pct = train_df[v_cols].isnull().mean()
    valid_v_cols = missing_pct[missing_pct < missing_threshold].index
    
    # Get top correlated features
    top_v_features = v_corr.loc[valid_v_cols].sort_values(ascending=False).head(top_n).index.tolist()
    
    print(f"Selected {len(top_v_features)} V features (top {top_n} by correlation, <{missing_threshold*100}% missing)")
    if top_v_features:
        print(f"Top 5 V features: {top_v_features[:5]}")
    
    return top_v_features

# First, select top V-features from the full dataset
top_v_features = select_top_v_features(train_data, top_n=150, missing_threshold=0.9)

# Then combine with core features
selected_features = features_common + top_v_features

# Now apply selection
train_data_selected = train_data[selected_features + ['isFraud']].copy()
test_data_selected = test_data[selected_features].copy()

print(f"After selection - Train: {train_data_selected.shape}, Test: {test_data_selected.shape}")

Selected 150 V features (top 150 by correlation, <90.0% missing)
Top 5 V features: ['V257', 'V246', 'V244', 'V242', 'V201']
After selection - Train: (590540, 194), Test: (506691, 193)


## 4. Transaction Amount Features


In [97]:
def create_transaction_amount_features(df, train_df=None):
    # Use training stats if provided (for test data)
    if train_df is not None:
        amount_stats = {
            'median': train_df['TransactionAmt'].median(),
            'q75': train_df['TransactionAmt'].quantile(0.75),
            'q25': train_df['TransactionAmt'].quantile(0.25),
            'mean': train_df['TransactionAmt'].mean(),
            'std': train_df['TransactionAmt'].std()
        }
        amount_stats['iqr'] = amount_stats['q75'] - amount_stats['q25']
    else:
        amount_stats = {
            'median': df['TransactionAmt'].median(),
            'q75': df['TransactionAmt'].quantile(0.75),
            'q25': df['TransactionAmt'].quantile(0.25),
            'mean': df['TransactionAmt'].mean(),
            'std': df['TransactionAmt'].std()
        }
        amount_stats['iqr'] = amount_stats['q75'] - amount_stats['q25']
    
    # 1. Log transform to handle extreme skew (skew=14.37, kurtosis=1123)
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    
    # 2. IQR-based outlier detection (11.26% outliers, 5.07% fraud rate)
    lower_bound = amount_stats['q25'] - 1.5 * amount_stats['iqr']
    upper_bound = amount_stats['q75'] + 1.5 * amount_stats['iqr']
    df['is_amount_outlier'] = ((df['TransactionAmt'] < lower_bound) | 
                              (df['TransactionAmt'] > upper_bound)).astype(int)
    
    # 3. Amount ratios (fraud median = 75 vs non-fraud median = 68.5)
    df['amount_to_median'] = df['TransactionAmt'] / amount_stats['median']
    df['amount_to_q75'] = df['TransactionAmt'] / amount_stats['q75']
    
    # 4. Statistical features
    df['amount_zscore'] = (df['TransactionAmt'] - amount_stats['mean']) / amount_stats['std']
    
    return df

# Apply to train and test
train_data_selected = create_transaction_amount_features(train_data_selected)
test_data_selected = create_transaction_amount_features(test_data_selected, train_data_selected)

# Verify fraud rates in outliers
if 'isFraud' in train_data_selected.columns:
    outlier_fraud_rate = train_data_selected[train_data_selected['is_amount_outlier'] == 1]['isFraud'].mean()
    normal_fraud_rate = train_data_selected[train_data_selected['is_amount_outlier'] == 0]['isFraud'].mean()
    print(f"Fraud rate in amount outliers: {outlier_fraud_rate*100:.2f}%")
    print(f"Fraud rate in normal range: {normal_fraud_rate*100:.2f}%")

Fraud rate in amount outliers: 5.07%
Fraud rate in normal range: 3.30%


## 5. Temporal Features from TransactionDT


In [98]:

def create_temporal_features(df):
    # Convert seconds to datetime-like features
    # 1. Basic time features
    df['hour'] = (df['TransactionDT'] // 3600) % 24
    df['dayofweek'] = (df['TransactionDT'] // (3600 * 24)) % 7
    
    # 2. Fraud peak hours (01:00–04:00 and 15:00–17:00 from EDA)
    df['is_fraud_peak_hour'] = (((df['hour'] >= 1) & (df['hour'] <= 4)) | 
                               ((df['hour'] >= 15) & (df['hour'] <= 17))).astype(int)
    
    # 3. Weekend flag (higher fraud on weekends from EDA)
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    
    # 4. Night flag (10 PM - 6 AM) - fraud increases during low-volume times
    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
    
    # 5. Cyclical encoding for hour (avoid artificial jumps)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    # 6. Cyclical encoding for dayofweek
    df['day_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    # 7. Late night weekend (high risk from EDA heatmap)
    df['is_late_night_weekend'] = ((df['is_night'] == 1) & (df['is_weekend'] == 1)).astype(int)
    
    return df

# Apply temporal feature engineering
train_data_selected = create_temporal_features(train_data_selected)
test_data_selected = create_temporal_features(test_data_selected)

# Verify temporal patterns
if 'isFraud' in train_data_selected.columns:
    hourly_fraud = train_data_selected.groupby('hour')['isFraud'].mean()
    peak_hour_fraud = train_data_selected[train_data_selected['is_fraud_peak_hour'] == 1]['isFraud'].mean()
    print(f"Fraud rate in peak hours: {peak_hour_fraud*100:.2f}%")
    print(f"Max fraud rate by hour: {hourly_fraud.max()*100:.2f}% at hour {hourly_fraud.idxmax()}")

Fraud rate in peak hours: 3.30%
Max fraud rate by hour: 10.61% at hour 7


## 6. ProductCD Features


In [99]:

def create_product_features(df):
    # 1. High-risk product flag (C=11.5% fraud, H=5.9% fraud from EDA)
    high_risk_products = ['C', 'H']
    df['is_high_risk_product'] = df['ProductCD'].isin(high_risk_products).astype(int)
    
    # 2. Medium-risk product flag
    medium_risk_products = ['R', 'S']  # R=4.0% fraud, S=3.4% fraud
    df['is_medium_risk_product'] = df['ProductCD'].isin(medium_risk_products).astype(int)
    
    # 3. Product risk score (based on EDA fraud rates)
    product_risk_scores = {
        'C': 3,  # Highest risk (11.5% fraud)
        'H': 2,  # High risk (5.9% fraud)
        'R': 1,  # Medium risk (4.0% fraud)
        'S': 1,  # Medium risk (3.4% fraud)
        'W': 0   # Lowest risk (2.2% fraud)
    }
    df['product_risk_score'] = df['ProductCD'].map(product_risk_scores).fillna(0)
    
    return df

# Apply product feature engineering
train_data_selected = create_product_features(train_data_selected)
test_data_selected = create_product_features(test_data_selected)

# Verify product risk patterns
if 'isFraud' in train_data_selected.columns:
    product_fraud_rates = train_data_selected.groupby('ProductCD')['isFraud'].mean()
    high_risk_fraud = train_data_selected[train_data_selected['is_high_risk_product'] == 1]['isFraud'].mean()
    print(f"Fraud rate in high-risk products: {high_risk_fraud*100:.2f}%")
    print("Actual fraud rates by product:")
    for product, rate in product_fraud_rates.items():
        print(f"  {product}: {rate*100:.2f}%")

Fraud rate in high-risk products: 9.44%
Actual fraud rates by product:
  C: 11.69%
  H: 4.77%
  R: 3.78%
  S: 5.90%
  W: 2.04%


## 7. Card Features Engineering


In [100]:

def create_card_features(df, train_df=None):
    
    # Calculate card1 frequency from training data
    if train_df is not None:
        card1_freq = train_df['card1'].value_counts()
    else:
        card1_freq = df['card1'].value_counts()
    
    # 1. Card frequency features (Real Estate Pattern - Frequency Encoding)
    df['card1_frequency'] = df['card1'].map(card1_freq).fillna(1)
    
    # 2. Single-use card flag (4.24% fraud vs 3.5% overall from EDA)
    df['is_single_use_card'] = (df['card1_frequency'] == 1).astype(int)
    
    # 3. High-frequency card flag (trusted customers)
    df['is_high_freq_card'] = (df['card1_frequency'] > 10).astype(int)
    
    # 4. NO BINNING - Using continuous frequency instead of categories
    # Binning removed: card_frequency_category
    
    # 5. Card type risk flags (from EDA: discover > mastercard > visa > amex)
    if 'card4' in df.columns:
        df['is_high_risk_card4'] = (df['card4'] == 'discover').astype(int)
        df['is_medium_risk_card4'] = df['card4'].isin(['mastercard', 'visa']).astype(int)
    
    # 6. Credit card flag (slightly higher fraud from EDA)
    if 'card6' in df.columns:
        df['is_credit_card'] = (df['card6'] == 'credit').astype(int)
        df['is_debit_card'] = (df['card6'] == 'debit').astype(int)
        df['is_debit_or_credit'] = df['card6'].isin(['credit', 'debit']).astype(int)
    
    # 7. Card feature interactions
    risk_cols = [col for col in ['is_high_risk_card4', 'is_credit_card', 'is_single_use_card'] if col in df.columns]
    if risk_cols:
        df['card_risk_combo'] = df[risk_cols].sum(axis=1)
    
    return df

# Apply card feature engineering
train_data_selected = create_card_features(train_data_selected)
test_data_selected = create_card_features(test_data_selected, train_data_selected)

# Verify card risk patterns
if 'isFraud' in train_data_selected.columns:
    single_use_fraud = train_data_selected[train_data_selected['is_single_use_card'] == 1]['isFraud'].mean()
    print(f"Fraud rate for single-use cards: {single_use_fraud*100:.2f}%")
    
    if 'card4' in train_data_selected.columns:
        card4_fraud = train_data_selected.groupby('card4')['isFraud'].mean()
        print("Fraud rates by card type:")
        for card_type, rate in card4_fraud.items():
            print(f"  {card_type}: {rate*100:.2f}%")

Fraud rate for single-use cards: 4.24%
Fraud rates by card type:
  american express: 2.87%
  discover: 7.73%
  mastercard: 3.43%
  visa: 3.48%


## 8. Count Features (C1-C14) Engineering


In [101]:

def create_count_features(df, train_df=None):
    
    # Calculate statistics from training data
    if train_df is not None:
        train_data_for_stats = train_df
    else:
        train_data_for_stats = df
    
    # Focus on top predictive C features from EDA: C1, C2, C12, C8, C9
    top_c_features = ['C1', 'C2', 'C12', 'C8', 'C9']
    
    for col in top_c_features:
        if col in df.columns:
            # Calculate IQR bounds
            Q1 = train_data_for_stats[col].quantile(0.25)
            Q3 = train_data_for_stats[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            
            # 1. Outlier flags (C1: 8.52% fraud, C2: 8.60% fraud in outliers from EDA)
            df[f'{col}_outlier'] = ((df[col] < lower) | (df[col] > upper)).astype(int)
            
            # 2. Log transform for skewed distributions
            if train_data_for_stats[col].skew() > 1:
                df[f'{col}_log'] = np.log1p(df[col] + 1)  # +1 to handle zeros
            
            # 3. Z-score normalization
            mean_val = train_data_for_stats[col].mean()
            std_val = train_data_for_stats[col].std()
            if std_val > 0:  # Avoid division by zero
                df[f'{col}_zscore'] = (df[col] - mean_val) / std_val
    
    # 4. Combined outlier flag
    outlier_cols = [f'{col}_outlier' for col in top_c_features if f'{col}_outlier' in df.columns]
    if outlier_cols:
        df['any_C_outlier'] = df[outlier_cols].max(axis=1)
    
    return df

# Apply count feature engineering
train_data_selected = create_count_features(train_data_selected)
test_data_selected = create_count_features(test_data_selected, train_data_selected)

# Verify count feature patterns
if 'isFraud' in train_data_selected.columns:
    if 'C1_outlier' in train_data_selected.columns:
        c1_outlier_fraud = train_data_selected[train_data_selected['C1_outlier'] == 1]['isFraud'].mean()
        c1_normal_fraud = train_data_selected[train_data_selected['C1_outlier'] == 0]['isFraud'].mean()
        print(f"C1 outlier fraud rate: {c1_outlier_fraud*100:.2f}% vs normal: {c1_normal_fraud*100:.2f}%")

C1 outlier fraud rate: 8.52% vs normal: 2.94%


## 9. Email Domain Features


In [102]:

def create_email_features(df, train_df=None):
    
    # Identify high-risk domains from training data
    high_risk_domains = set()
    
    if train_df is not None and 'isFraud' in train_df.columns:
        for email_col in ['P_emaildomain', 'R_emaildomain']:
            if email_col in train_df.columns:
                fraud_rates = train_df.groupby(email_col)['isFraud'].mean()
                # Domains with fraud rate > 2x overall average
                high_risk = fraud_rates[fraud_rates > train_df['isFraud'].mean() * 2].index
                high_risk_domains.update(high_risk)
                print(f"Found {len(high_risk)} high-risk domains in {email_col}")
    else:
        # Default high-risk domains from EDA
        high_risk_domains = {'protonmail.com', 'mail.com', 'outlook.es', 'netzero.net', 'aim.com'}
        print("Using default high-risk domains from EDA")
    
    # Advanced email provider extraction (Real Estate Pattern - Text Extraction)
    def extract_email_provider(email_domain):
        if pd.isna(email_domain): return "missing"
        email_domain = str(email_domain).lower()
        if 'gmail' in email_domain: return 'gmail'
        if 'yahoo' in email_domain: return 'yahoo' 
        if 'hotmail' in email_domain: return 'hotmail'
        if 'protonmail' in email_domain: return 'protonmail'
        if 'outlook' in email_domain: return 'outlook'
        if 'aol' in email_domain: return 'aol'
        if 'icloud' in email_domain: return 'icloud'
        return 'other'
    
    for email_col in ['P_emaildomain', 'R_emaildomain']:
        if email_col in df.columns:
            col_suffix = email_col.replace('emaildomain', '').replace('_', '')
            
            # 1. Email provider extraction (Real Estate Pattern)
            df[f'email_provider{col_suffix}'] = df[email_col].apply(extract_email_provider)
            
            # 2. High-risk domain flag
            df[f'is_high_risk{col_suffix}'] = df[email_col].isin(high_risk_domains).astype(int)
            
            # 3. Extremely high-risk flags (from EDA: protonmail=40.79-95.12% fraud)
            df[f'is_protonmail{col_suffix}'] = df[email_col].str.contains('protonmail', na=False).astype(int)
            df[f'is_mail_com{col_suffix}'] = df[email_col].str.contains('mail.com', na=False).astype(int)
            
            # 4. Missing email flag
            df[f'is_missing{col_suffix}'] = df[email_col].isna().astype(int)
            
            # 5. Popular email providers (lower risk)
            popular_domains = ['gmail.com', 'yahoo.com', 'hotmail.com', 'aol.com', 'icloud.com']
            df[f'is_popular{col_suffix}'] = df[email_col].isin(popular_domains).astype(int)
            
            # 6. Anonymous email flag
            df[f'is_anonymous{col_suffix}'] = (df[email_col] == 'anonymous.com').astype(int)
    
    # 7. Email match flag
    if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
        df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        df['email_match'] = df['email_match'].fillna(0)  # If either is missing, no match
    
    return df

# Apply email feature engineering
train_data_selected = create_email_features(train_data_selected, train_data)
test_data_selected = create_email_features(test_data_selected, train_data)

# Verify email risk patterns
if 'isFraud' in train_data_selected.columns:
    if 'is_high_riskP' in train_data_selected.columns:
        high_risk_email_fraud = train_data_selected[train_data_selected['is_high_riskP'] == 1]['isFraud'].mean()
        print(f"Fraud rate in high-risk email domains: {high_risk_email_fraud*100:.2f}%")
    
    if 'is_protonmailR' in train_data_selected.columns:
        protonmail_fraud = train_data_selected[train_data_selected['is_protonmailR'] == 1]['isFraud'].mean()
        print(f"Fraud rate for protonmail (R): {protonmail_fraud*100:.2f}%")



Found 5 high-risk domains in P_emaildomain
Found 9 high-risk domains in R_emaildomain
Found 5 high-risk domains in P_emaildomain
Found 9 high-risk domains in R_emaildomain
Fraud rate in high-risk email domains: 4.62%
Fraud rate for protonmail (R): 95.12%


## 10. Identity Features Engineering


In [103]:

def create_identity_features(df):
    # Clean OS information
    def clean_os(x):
        if pd.isna(x): return "Missing"
        x = str(x).split()[0]
        if x in ['Android', 'iOS', 'Mac', 'Windows', 'Linux']: return x
        return "Other"
    
    # Clean Browser information  
    def clean_browser(x):
        if pd.isna(x): return "Missing"
        x = str(x).lower()
        if 'chrome' in x: return 'Chrome'
        if 'safari' in x: return 'Safari' 
        if 'firefox' in x: return 'Firefox'
        if 'samsung' in x or 'samsungbrowser' in x: return 'Samsung'
        if 'edge' in x or 'ie ' in x: return 'IE/Edge'
        if 'android' in x: return 'Android Browser'
        return "Other"
    
    # Advanced Device Info Extraction (Real Estate Pattern)
    def extract_device_os(device_info):
        if pd.isna(device_info): return "Missing"
        device_info = str(device_info).lower()
        if 'windows' in device_info: return 'Windows'
        if 'android' in device_info: return 'Android'
        if 'ios' in device_info or 'iphone' in device_info: return 'iOS'
        if 'mac' in device_info: return 'Mac'
        if 'linux' in device_info: return 'Linux'
        return 'Other'
    
    # 1. OS features (id_30) - Other=36%, Android=8.6%, Linux=7.48% fraud from EDA
    if 'id_30' in df.columns:
        df['OS'] = df['id_30'].apply(clean_os)
        df['is_high_risk_OS'] = df['OS'].isin(['Other', 'Android', 'Linux']).astype(int)
        df['is_windows_OS'] = (df['OS'] == 'Windows').astype(int)
        df['is_mobile_OS'] = df['OS'].isin(['Android', 'iOS']).astype(int)
        df['OS_missing'] = df['id_30'].isna().astype(int)
    
    # 2. Browser features (id_31) - Other=21.56%, Android Browser=20.42% fraud from EDA
    if 'id_31' in df.columns:
        df['Browser'] = df['id_31'].apply(clean_browser)
        df['is_high_risk_browser'] = df['Browser'].isin(['Other', 'Android Browser']).astype(int)
        df['is_chrome_browser'] = (df['Browser'] == 'Chrome').astype(int)
        df['is_safari_browser'] = (df['Browser'] == 'Safari').astype(int)
        df['browser_missing'] = df['id_31'].isna().astype(int)
    
    # 3. Device type features
    if 'DeviceType' in df.columns:
        df['is_mobile_device'] = (df['DeviceType'] == 'mobile').astype(int)
        df['is_desktop_device'] = (df['DeviceType'] == 'desktop').astype(int)
        df['DeviceType_missing'] = df['DeviceType'].isna().astype(int)
    
    # 4. Advanced Device Info parsing (Real Estate Pattern)
    if 'DeviceInfo' in df.columns:
        df['device_os'] = df['DeviceInfo'].apply(extract_device_os)
        df['device_os_missing'] = df['DeviceInfo'].isna().astype(int)
    
    # 5. Boolean identity flags (id_35-id_38) - False = higher fraud from EDA
    for bool_col in ['id_35', 'id_36', 'id_37', 'id_38']:
        if bool_col in df.columns:
            df[f'{bool_col}_is_F'] = (df[bool_col] == 'F').astype(int)
            df[f'{bool_col}_is_T'] = (df[bool_col] == 'T').astype(int)
            df[f'{bool_col}_missing'] = df[bool_col].isna().astype(int)
    
    # 6. Combined trust score (more F's = less trusted)
    trust_cols = [f'{col}_is_F' for col in ['id_35', 'id_36', 'id_37', 'id_38'] 
                 if f'{col}_is_F' in df.columns]
    if trust_cols:
        df['device_trust_score'] = df[trust_cols].sum(axis=1)
    
    # 7. Screen resolution features (id_33)
    if 'id_33' in df.columns:
        df['resolution_missing'] = df['id_33'].isna().astype(int)
        # Common resolutions might be lower risk
        common_resolutions = ['1920x1080', '1366x768', '1334x750', '2208x1242', '1440x900']
        df['is_common_resolution'] = df['id_33'].isin(common_resolutions).astype(int)
    
    return df

# Apply identity feature engineering
train_data_selected = create_identity_features(train_data_selected)
test_data_selected = create_identity_features(test_data_selected)

# Verify identity risk patterns
if 'isFraud' in train_data_selected.columns:
    if 'is_high_risk_OS' in train_data_selected.columns:
        high_risk_os_fraud = train_data_selected[train_data_selected['is_high_risk_OS'] == 1]['isFraud'].mean()
        print(f"Fraud rate for high-risk OS: {high_risk_os_fraud*100:.2f}%")
    
    if 'is_high_risk_browser' in train_data_selected.columns:
        high_risk_browser_fraud = train_data_selected[train_data_selected['is_high_risk_browser'] == 1]['isFraud'].mean()
        print(f"Fraud rate for high-risk browsers: {high_risk_browser_fraud*100:.2f}%")

## 11. Missingness Indicators


In [104]:

def create_missingness_features(df):
    
    # Feature groups to create missingness indicators for
    feature_groups = {
        'D_features': ['D1', 'D2', 'D3', 'D4', 'D8', 'D10', 'D15'],
        'card_features': ['card2', 'card3', 'card5', 'card6'],
        'identity_features': [f'id_{i:02d}' for i in range(1, 12)] + 
                           ['id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38'],
        'email_features': ['P_emaildomain', 'R_emaildomain'],
        'M_features': ['M1', 'M2', 'M3', 'M4', 'M5', 'M6'],
        'address_features': ['addr1', 'addr2']
    }
    
    for group_name, features in feature_groups.items():
        available_features = [f for f in features if f in df.columns]
        if available_features:
            df[f'missing_{group_name}'] = df[available_features].isnull().mean(axis=1)
    
    # Overall missingness (excluding TransactionID and target)
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    numerical_cols = [col for col in numerical_cols if col not in ['TransactionID', 'isFraud']]
    if numerical_cols:
        df['missing_overall'] = df[numerical_cols].isnull().mean(axis=1)
    
    # High missingness flag (>50% missing)
    df['is_high_missing'] = (df['missing_overall'] > 0.5).astype(int)
    
    return df

# Apply missingness feature engineering
train_data_selected = create_missingness_features(train_data_selected)
test_data_selected = create_missingness_features(test_data_selected)

print("Created Missingness Features:")
print("✓ Group-wise missingness (D, card, identity, email, M, address features)")
print("✓ Overall missingness percentage")
print("✓ High missingness flag (>50% missing)")

# Verify missingness patterns
if 'isFraud' in train_data_selected.columns:
    fraud_missing = train_data_selected[train_data_selected['isFraud'] == 1]['missing_overall'].mean()
    non_fraud_missing = train_data_selected[train_data_selected['isFraud'] == 0]['missing_overall'].mean()
    print(f"Average missingness - Fraud: {fraud_missing*100:.1f}%, Non-Fraud: {non_fraud_missing*100:.1f}%")
    print(f"Difference: {abs(fraud_missing - non_fraud_missing)*100:.1f}% (fraud has fewer missing values)")

Created Missingness Features:
✓ Group-wise missingness (D, card, identity, email, M, address features)
✓ Overall missingness percentage
✓ High missingness flag (>50% missing)
Average missingness - Fraud: 24.1%, Non-Fraud: 28.1%
Difference: 4.0% (fraud has fewer missing values)


## 12. Advanced Feature Engineering


### 12.a). COMPOSITE FRAUD RISK SCORE

In [105]:

def create_fraud_risk_score(df):
    # Fraud risk weights based on EDA findings (Real Estate Pattern - Weighted Scoring)
    fraud_risk_weights = {
        'is_protonmailP': 10,        # Extreme risk: 40-95% fraud
        'is_protonmailR': 10,        # Extreme risk: 40-95% fraud  
        'is_single_use_card': 8,     # High risk: 4.24% fraud
        'is_high_risk_product': 7,   # Product C & H: 11.5-5.9% fraud
        'is_amount_outlier': 6,      # Transaction amount anomalies
        'is_high_risk_OS': 7,        # Device risk: Other=36%, Android=8.6% fraud
        'is_high_risk_browser': 7,   # Browser risk: Android Browser=20.42% fraud
        'C1_outlier': 6,             # Count feature anomalies: 8.52% fraud
        'C2_outlier': 6,             # Count feature anomalies: 8.60% fraud
        'is_fraud_peak_hour': 5,     # Temporal patterns
        'is_high_risk_card4': 5,     # Card type risk
        'is_credit_card': 4,         # Slightly higher fraud
        'email_match': 3,            # Lower risk factor
        'is_weekend': 4,             # Temporal risk
    }
    
    risk_score = 0
    weights_used = 0
    
    for feature, weight in fraud_risk_weights.items():
        if feature in df.columns:
            risk_score += df[feature] * weight
            weights_used += weight
    
    # Normalize to 0-100 scale
    if weights_used > 0:
        df['composite_fraud_risk'] = (risk_score / weights_used) * 100
    else:
        df['composite_fraud_risk'] = 0
    
    return df

# Apply composite risk scoring
train_data_selected = create_fraud_risk_score(train_data_selected)
test_data_selected = create_fraud_risk_score(test_data_selected)


### 12.b). BEHAVIORAL CLUSTERING

In [108]:
def create_behavioral_clusters_train(df, n_clusters=4):
    # Ensure target exists → this must be training data
    if 'isFraud' not in df.columns:
        raise ValueError("Training data must contain 'isFraud' column.")

    # Behavioral features for clustering
    behavioral_features = [
        'TransactionAmt',
        'hour',
        'dayofweek',
        'card1_frequency',
        'product_risk_score',
        'composite_fraud_risk'
    ]

    # Use only available features
    available_features = [f for f in behavioral_features if f in df.columns]

    # If not enough features → fallback
    if len(available_features) < 3:
        print("Not enough features for clustering. Assigning cluster = 0")
        df['behavioral_cluster'] = 0
        return df, None, None

    # Prepare data
    X = df[available_features].fillna(-999)

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # KMeans clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    df['behavioral_cluster'] = kmeans.fit_predict(X_scaled)

    # Analyze fraud rate per cluster
    print("\nFraud rates by behavioral cluster (Train Only):")
    cluster_fraud_rates = df.groupby('behavioral_cluster')['isFraud'].mean()
    for cluster, rate in cluster_fraud_rates.items():
        print(f"  Cluster {cluster}: {rate*100:.2f}%")

    return df, scaler, kmeans

def assign_behavioral_clusters_test(df, train_df):
    # Example: if TransactionAmt is large → higher risk cluster
    if 'TransactionAmt' in df.columns:

        high_amt_threshold = train_df['TransactionAmt'].quantile(0.90)
        medium_amt_threshold = train_df['TransactionAmt'].quantile(0.70)

        df.loc[df['TransactionAmt'] > high_amt_threshold, 'behavioral_cluster'] = 2
        df.loc[(df['TransactionAmt'] > medium_amt_threshold) &
               (df['TransactionAmt'] <= high_amt_threshold), 'behavioral_cluster'] = 1

    return df

train_data_selected, cluster_scaler, cluster_model = create_behavioral_clusters_train(train_data_selected)

test_data_selected = assign_behavioral_clusters_test(test_data_selected, train_data_selected)



Fraud rates by behavioral cluster (Train Only):
  Cluster 0: 2.56%
  Cluster 1: 1.95%
  Cluster 2: 9.73%
  Cluster 3: 2.20%


### 12C. ADVANCED INTERACTION FEATURES

In [109]:
def create_interaction_features(df):
    # High-risk combinations (Real Estate Pattern - Feature Interactions)
    if all(col in df.columns for col in ['is_single_use_card', 'is_high_risk_product']):
        df['single_use_high_risk_product'] = df['is_single_use_card'] * df['is_high_risk_product']
    
    if all(col in df.columns for col in ['is_protonmailP', 'is_amount_outlier']):
        df['protonmail_large_amount'] = df['is_protonmailP'] * df['is_amount_outlier']
    
    if all(col in df.columns for col in ['is_fraud_peak_hour', 'is_weekend']):
        df['peak_hour_weekend'] = df['is_fraud_peak_hour'] * df['is_weekend']
    
    # Device-risk combinations
    if all(col in df.columns for col in ['is_high_risk_OS', 'is_high_risk_browser']):
        df['high_risk_device_combo'] = df['is_high_risk_OS'] * df['is_high_risk_browser']
    
    return df

In [110]:
# Apply interaction features
train_data_selected = create_interaction_features(train_data_selected)
test_data_selected = create_interaction_features(test_data_selected)

## 13. Final Feature Encoding & Preparation

In [111]:
def encode_categorical_features(df, label_encoders=None, frequency_encoders=None):
    
    if label_encoders is None:
        label_encoders = {}
    if frequency_encoders is None:
        frequency_encoders = {}
    
    categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
    
    for col in categorical_columns:
        if col not in ['TransactionID', 'isFraud']:
            n_unique = df[col].nunique()
            
            if n_unique <= 10:
                # Low cardinality - use label encoding
                if col not in label_encoders:
                    le = LabelEncoder()
                    # Combine with a placeholder for unknown values
                    unique_vals = df[col].fillna('unknown').unique()
                    le.fit(unique_vals)
                    label_encoders[col] = le
                
                df[f'{col}_encoded'] = label_encoders[col].transform(df[col].fillna('unknown'))
                
            else:
                # High cardinality - use frequency encoding (Real Estate Pattern)
                if col not in frequency_encoders:
                    freq_map = df[col].value_counts().to_dict()
                    frequency_encoders[col] = freq_map
                
                df[f'{col}_freq'] = df[col].map(frequency_encoders[col])
                df[f'{col}_freq'] = df[f'{col}_freq'].fillna(1)  # Treat unseen as rare
    
    return df, label_encoders, frequency_encoders

In [112]:
# Encode categorical features
train_data_encoded, label_encoders, frequency_encoders = encode_categorical_features(train_data_selected)
test_data_encoded, _, _ = encode_categorical_features(test_data_selected, label_encoders, frequency_encoders)

print(f"Encoded {len(label_encoders)} categorical columns with label encoding")
print(f"Encoded {len(frequency_encoders)} categorical columns with frequency encoding")

Encoded 13 categorical columns with label encoding
Encoded 3 categorical columns with frequency encoding


In [113]:
# Drop original categorical columns (keep encoded versions)
categorical_cols = train_data_encoded.select_dtypes(include=['object']).columns.tolist()
cols_to_drop = [col for col in categorical_cols if col not in ['TransactionID', 'isFraud']]
train_data_encoded = train_data_encoded.drop(columns=cols_to_drop)
test_data_encoded = test_data_encoded.drop(columns=[col for col in cols_to_drop if col in test_data_encoded.columns])

print(f"After encoding - Train: {train_data_encoded.shape}, Test: {test_data_encoded.shape}")

After encoding - Train: (590540, 269), Test: (506691, 268)


In [114]:
# Handle remaining missing values (use -999 for tree models)
train_data_encoded = train_data_encoded.fillna(-999)
test_data_encoded = test_data_encoded.fillna(-999)

print("Final data preparation complete!")

Final data preparation complete!


In [115]:
# ADD THIS: Validate that all engineered features exist in both datasets
def validate_features(train_df, test_df):
    """Validate that all features exist in both datasets"""
    train_features = set(train_df.columns) - {'TransactionID', 'isFraud'}
    test_features = set(test_df.columns) - {'TransactionID'}
    
    missing_in_test = train_features - test_features
    missing_in_train = test_features - train_features
    
    if missing_in_test:
        print(f"❌ WARNING: {len(missing_in_test)} features missing in test data: {list(missing_in_test)[:5]}...")
    
    if missing_in_train:
        print(f"❌ WARNING: {len(missing_in_train)} features missing in train data: {list(missing_in_train)[:5]}...")
    
    if not missing_in_test and not missing_in_train:
        print("✅ All features consistent between train and test")
    
    return train_features & test_features  # Return common features

common_features = validate_features(train_data_encoded, test_data_encoded)

✅ All features consistent between train and test


In [116]:
# ADD THIS: Comprehensive validation before finishing
def final_validation(train_df, test_df):
    """Comprehensive validation of engineered features"""
    
    print("\n" + "="*50)
    print("FINAL VALIDATION")
    print("="*50)
    
    # 1. Check for NaN values
    train_nan = train_df.isnull().sum().sum()
    test_nan = test_df.isnull().sum().sum()
    print(f"NaN values - Train: {train_nan}, Test: {test_nan}")
    
    # 2. Check feature counts
    train_features = len([col for col in train_df.columns if col not in ['TransactionID', 'isFraud']])
    test_features = len([col for col in test_df.columns if col != 'TransactionID'])
    print(f"Feature count - Train: {train_features}, Test: {test_features}")
    
    # 3. Check data types
    object_cols_train = train_df.select_dtypes(include=['object']).columns.tolist()
    object_cols_test = test_df.select_dtypes(include=['object']).columns.tolist()
    print(f"Object columns - Train: {len(object_cols_train)}, Test: {len(object_cols_test)}")
    
    # 4. Check for infinite values
    train_inf = np.isinf(train_df.select_dtypes(include=[np.number])).sum().sum()
    test_inf = np.isinf(test_df.select_dtypes(include=[np.number])).sum().sum()
    print(f"Infinite values - Train: {train_inf}, Test: {test_inf}")
    
    # 5. Validate key engineered features exist
    key_features = ['composite_fraud_risk', 'behavioral_cluster', 'TransactionAmt_log']
    missing_key = [f for f in key_features if f not in train_df.columns or f not in test_df.columns]
    if missing_key:
        print(f"❌ Missing key features: {missing_key}")
    else:
        print("✅ All key engineered features present")
    
    return train_features == test_features and train_nan == 0 and test_nan == 0

# Run final validation
is_valid = final_validation(train_data_encoded, test_data_encoded)

if is_valid:
    print("\n🎉 ALL VALIDATIONS PASSED! Feature engineering completed successfully.")
else:
    print("\n⚠️  Some validations failed. Please review the warnings above.")


FINAL VALIDATION
NaN values - Train: 0, Test: 0
Feature count - Train: 267, Test: 267
Object columns - Train: 0, Test: 0
Infinite values - Train: 0, Test: 0
✅ All key engineered features present

🎉 ALL VALIDATIONS PASSED! Feature engineering completed successfully.


In [78]:
train_data_encoded.head()

,C5,D1,D15,addr2,addr1,D2,card1,C9,TransactionID,C13,...,card6_encoded,M4_encoded,M5_encoded,P_emaildomain_freq,DeviceType_encoded,M2_encoded,M1_encoded,email_providerP_encoded,email_providerR_encoded,device_os_encoded
0,0.0,14.0,0.0,87.0,315.0,-999.0,13926,1.0,2987000,1.0,...,1,2,0,1.0,2,1,1,4,4,3
1,0.0,0.0,0.0,87.0,325.0,-999.0,2755,0.0,2987001,1.0,...,1,0,1,228355.0,2,2,2,1,4,3
2,0.0,0.0,315.0,87.0,330.0,-999.0,4663,1.0,2987002,1.0,...,2,0,0,5096.0,2,1,1,6,4,3
3,0.0,112.0,111.0,87.0,476.0,112.0,18132,1.0,2987003,25.0,...,2,0,1,100934.0,2,2,2,8,4,3
4,0.0,0.0,-999.0,87.0,420.0,-999.0,4497,0.0,2987004,1.0,...,1,3,2,228355.0,1,2,2,1,4,4


In [79]:
train_data_encoded.shape

(590540, 262)

In [67]:
train_data_encoded.isnull().sum()

C5                         0
D1                         0
D15                        0
addr2                      0
addr1                      0
                          ..
M2_encoded                 0
M1_encoded                 0
email_providerP_encoded    0
email_providerR_encoded    0
device_os_encoded          0
Length: 262, dtype: int64

In [80]:
test_data_encoded.head()

,C5,D1,D15,addr2,addr1,D2,card1,C9,TransactionID,C13,...,card6_encoded,M4_encoded,M5_encoded,P_emaildomain_freq,DeviceType_encoded,M2_encoded,M1_encoded,email_providerP_encoded,email_providerR_encoded,device_os_encoded
0,3.0,419.0,409.0,87.0,170.0,419.0,10409,6.0,3663549,115.0,...,2,3,2,228355.0,2,1,1,1,4,3
1,0.0,149.0,634.0,87.0,299.0,149.0,4272,2.0,3663550,12.0,...,2,0,2,28289.0,2,0,1,0,4,3
2,0.0,137.0,97.0,87.0,472.0,137.0,4476,4.0,3663551,22.0,...,2,0,0,45250.0,2,1,1,2,4,3
3,1.0,42.0,242.0,87.0,205.0,42.0,10989,2.0,3663552,7.0,...,2,3,2,228355.0,2,1,1,1,4,3
4,2.0,22.0,22.0,87.0,264.0,22.0,18018,5.0,3663553,14.0,...,2,3,2,228355.0,2,1,1,1,4,3


In [81]:
test_data_encoded.shape

(506691, 261)

In [82]:
test_data_encoded.isnull().sum()

C5                         0
D1                         0
D15                        0
addr2                      0
addr1                      0
                          ..
M2_encoded                 0
M1_encoded                 0
email_providerP_encoded    0
email_providerR_encoded    0
device_os_encoded          0
Length: 261, dtype: int64

## 14. Save Engineered Features for Feature Selection

In [118]:
# Save fully engineered datasets (before final feature selection)
train_data_encoded.to_parquet(f"{DATA_FEATURES_PATH}train_features_fully_engineered.parquet", index=False)
test_data_encoded.to_parquet(f"{DATA_FEATURES_PATH}test_features_fully_engineered.parquet", index=False)

print("Saved fully engineered features for feature selection:")
print(f"Train: {DATA_FEATURES_PATH}train_features_fully_engineered.parquet")
print(f"Test: {DATA_FEATURES_PATH}test_features_fully_engineered.parquet")

# Feature summary
if 'isFraud' in train_data_encoded.columns:
    feature_cols = [col for col in train_data_encoded.columns if col not in ['TransactionID', 'isFraud']]
else:
    feature_cols = [col for col in train_data_encoded.columns if col != 'TransactionID']

print(f"\nFinal engineered feature count: {len(feature_cols)}")

Saved fully engineered features for feature selection:
Train: ../data/processed/features/train_features_fully_engineered.parquet
Test: ../data/processed/features/test_features_fully_engineered.parquet

Final engineered feature count: 267


In [120]:
# Step 1 — Build all categories except "Other"
feature_categories = {
    'Transaction Amount': [col for col in feature_cols if 'amount' in col.lower() or 'transactionamt' in col.lower()],
    'Temporal': [col for col in feature_cols if any(x in col.lower() for x in ['hour', 'day', 'weekend', 'night', 'peak'])],
    'Product': [col for col in feature_cols if 'product' in col.lower()],
    'Card': [col for col in feature_cols if 'card' in col.lower()],
    'Count Features': [col for col in feature_cols if col.startswith('C') and len(col) <= 3],
    'Email': [col for col in feature_cols if 'email' in col.lower()],
    'Identity': [col for col in feature_cols if 'id_' in col or any(x in col.lower() for x in ['os', 'browser', 'device'])],
    'Missingness': [col for col in feature_cols if 'missing' in col.lower()],
    'Advanced Features': [col for col in feature_cols if any(x in col.lower() for x in ['composite', 'cluster', 'risk', 'interaction'])],
    'V-Features': [col for col in feature_cols if col.startswith('V')],
}

# Step 2 — Add "Other" category
all_used_features = {item for sublist in feature_categories.values() for item in sublist}
feature_categories['Other'] = [col for col in feature_cols if col not in all_used_features]

# Print summary
print("\nFeature Categories:")
for category, features in feature_categories.items():
    if features:
        print(f"  {category}: {len(features)} features")



Feature Categories:
  Transaction Amount: 7 features
  Temporal: 11 features
  Product: 5 features
  Card: 15 features
  Count Features: 14 features
  Email: 6 features
  Identity: 10 features
  Missingness: 11 features
  Advanced Features: 11 features
  V-Features: 150 features
  Other: 41 features


## 15. Final Summary Report

In [121]:
# Calculate original feature count dynamically
original_features = len([c for c in train_data.columns if c not in ['TransactionID', 'isFraud']])

# Calculate final feature count after full engineering
final_features = len(feature_cols)

# Calculate engineered features
engineered_features = final_features - original_features

summary_stats = {
    'Original Features': original_features,
    'Engineered Features Added': engineered_features,
    'Final Feature Count': final_features,
    'Feature Increase': f"{(final_features / original_features - 1) * 100:+.1f}%",
    'Memory Usage': f"{train_data_encoded.memory_usage(deep=True).sum() / 1024**2:.1f} MB",
    'Fraud Rate': f"{train_data_encoded['isFraud'].mean() * 100:.2f}%",
    'Training Samples': f"{train_data_encoded.shape[0]:,}",
    'Test Samples': f"{test_data_encoded.shape[0]:,}"
}

print("\nSummary Statistics:")
for key, value in summary_stats.items():
    print(f"  {key:25}: {value}")


Summary Statistics:
  Original Features        : 432
  Engineered Features Added: -165
  Final Feature Count      : 267
  Feature Increase         : -38.2%
  Memory Usage             : 1088.1 MB
  Fraud Rate               : 3.50%
  Training Samples         : 590,540
  Test Samples             : 506,691
